# LeetCode 2974 — Minimum Number Game bằng Python

Link đề: https://leetcode.com/problems/minimum-number-game/ · Độ khó: **Easy** · Chủ đề: Array, Sorting, Heap (Priority Queue), Simulation

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def numberGame(self, nums: List[int]) -> List[int]:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: hai người chơi lần lượt **lấy phần tử nhỏ nhất** khỏi `nums`; mỗi vòng Alice lấy trước, Bob lấy sau, nhưng **Bob thêm phần tử của mình vào `arr` trước, rồi mới đến Alice**.
2. Biết **mô phỏng** một trò chơi bằng vòng `while`, dùng `min`, `remove`, `append`.
3. Nhận ra **bước đột phá**: "lấy nhỏ nhất liên tục" chính là **duyệt theo thứ tự đã sắp xếp**; kết quả là mảng đã sắp xếp với **mỗi cặp liền kề đổi chỗ**.
4. Đi từ cách **mô phỏng bằng `min` và `remove`** → **sắp xếp rồi đổi chỗ từng cặp** → **min-heap `heapq`** → **sắp xếp đếm (counting sort)** vì giá trị `<= 100`.
5. Tránh các lỗi kinh điển: thêm phần tử của Alice trước Bob (sai thứ tự), quên sắp xếp, đổi chỗ các cặp bị lệch (`range(n - 1)`), và vô tình sửa luôn mảng đầu vào bằng `nums.sort()`.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` có độ dài **chẵn** (đánh chỉ số từ `0`). Alice và Bob chơi một trò chơi với `nums` theo các quy tắc sau; ban đầu có một mảng rỗng `arr`.

Mỗi **vòng** gồm:

1. **Alice** lấy ra (xóa) **phần tử nhỏ nhất** khỏi `nums`;
2. sau đó **Bob** lấy ra (xóa) **phần tử nhỏ nhất** khỏi `nums` (lúc này đã mất phần tử của Alice);
3. **Bob** thêm phần tử mà Bob vừa lấy vào cuối `arr` **trước**;
4. rồi **Alice** thêm phần tử mà Alice vừa lấy vào cuối `arr`.

Trò chơi kết thúc khi `nums` rỗng. Trả về mảng `arr`.

**Ví dụ 1**

- Input: `nums = [5, 4, 2, 3]`
- Output: `[3, 2, 5, 4]`
- Giải thích: vòng 1: Alice lấy `2`, Bob lấy `3`; Bob thêm `3` rồi Alice thêm `2`, `arr = [3, 2]`. Vòng 2: `nums = [5, 4]`, Alice lấy `4`, Bob lấy `5`; Bob thêm `5` rồi Alice thêm `4`, `arr = [3, 2, 5, 4]`.

**Ví dụ 2**

- Input: `nums = [2, 5]`
- Output: `[5, 2]`
- Giải thích: Alice lấy `2`, Bob lấy `5`; Bob thêm `5`, Alice thêm `2`, `arr = [5, 2]`.

**Ràng buộc:** `2 <= nums.length <= 100`; `1 <= nums[i] <= 100`; `nums.length` là số **chẵn**.

In [1]:
from typing import List
import heapq
import random

examples = [
    ([5, 4, 2, 3], [3, 2, 5, 4]),
    ([2, 5], [5, 2]),
    ([1, 1, 1, 1], [1, 1, 1, 1]),
]
for nums, expected in examples:
    print("nums =", nums, "-> kỳ vọng", expected)

nums = [5, 4, 2, 3] -> kỳ vọng [3, 2, 5, 4]
nums = [2, 5] -> kỳ vọng [5, 2]
nums = [1, 1, 1, 1] -> kỳ vọng [1, 1, 1, 1]


## 3. Kiến thức nền

### 3.1. `min`, `remove`, `append` trên list

- `min(a)` trả về phần tử nhỏ nhất;
- `a.remove(x)` xóa **lần xuất hiện đầu tiên** của `x` (báo lỗi nếu không có);
- `a.append(x)` thêm `x` vào cuối.

Mỗi lệnh `min` và `remove` đều quét danh sách nên tốn `O(len(a))`.

In [2]:
a = [5, 4, 2, 3, 2]
m = min(a)
print(m)                       # 2
a.remove(m)                    # xóa chỉ MỘT số 2 (lần đầu)
print(a)                       # [5, 4, 3, 2]
a.append(9)
print(a)                       # [5, 4, 3, 2, 9]

2
[5, 4, 3, 2]
[5, 4, 3, 2, 9]


### 3.2. Sắp xếp: `sorted` và `list.sort`

`sorted(a)` trả về **danh sách mới** đã sắp xếp tăng dần, **không đổi** `a`. `a.sort()` **sắp xếp ngay trên `a`** và trả về `None`. Với dữ liệu của người gọi, nên dùng `sorted` (hoặc sao chép trước) để tránh tác dụng phụ.

In [3]:
a = [5, 4, 2, 3]
b = sorted(a)
print(a, b)                    # [5, 4, 2, 3] [2, 3, 4, 5]
c = a.sort()
print(a, c)                    # [2, 3, 4, 5] None  <- a bị đổi, c là None

[5, 4, 2, 3] [2, 3, 4, 5]
[2, 3, 4, 5] None


### 3.3. Cắt lát có bước: `a[::2]` và `a[1::2]`

`a[::2]` lấy các phần tử ở chỉ số **chẵn** (`0, 2, 4, ...`), `a[1::2]` lấy các phần tử ở chỉ số **lẻ**. Hai lát cắt này "tách" dãy thành hai nửa xen kẽ.

In [4]:
a = [2, 3, 4, 5, 6, 7]
print(a[::2])                  # [2, 4, 6]
print(a[1::2])                 # [3, 5, 7]

[2, 4, 6]
[3, 5, 7]


### 3.4. Gán vào lát cắt có bước để đổi chỗ từng cặp

Python cho phép **gán đồng thời**: `a[::2], a[1::2] = a[1::2], a[::2]`. Vế phải được tính xong trước, rồi gán: các vị trí chẵn nhận các giá trị lẻ cũ và ngược lại, tức **đổi chỗ từng cặp liền kề** `(a[0], a[1])`, `(a[2], a[3])`, ... Với độ dài chẵn thì hai lát cắt cùng độ dài nên gán hợp lệ.

In [5]:
a = [2, 3, 4, 5, 6, 7]
a[::2], a[1::2] = a[1::2], a[::2]
print(a)                       # [3, 2, 5, 4, 7, 6]
# Cách viết dài tương đương
b = [2, 3, 4, 5, 6, 7]
for i in range(0, len(b), 2):
    b[i], b[i + 1] = b[i + 1], b[i]
print(b == a)                  # True

[3, 2, 5, 4, 7, 6]
True


### 3.5. Hàng đợi ưu tiên: `heapq`

`heapq` quản lý một **min-heap** trên list: `heapq.heapify(a)` biến `a` thành heap (`O(n)`), `heapq.heappop(a)` lấy phần tử nhỏ nhất (`O(log n)`), `heapq.heappush(a, x)` thêm phần tử. Dùng khi cần "lấy nhỏ nhất" nhiều lần.

In [6]:
h = [5, 4, 2, 3]
heapq.heapify(h)
print(heapq.heappop(h), heapq.heappop(h), heapq.heappop(h), heapq.heappop(h))   # 2 3 4 5

2 3 4 5


### 3.6. Mảng đếm (counting sort) khi giá trị nhỏ

Vì `1 <= nums[i] <= 100`, ta có thể đếm `cnt[v]` rồi in ra mỗi `v` đúng `cnt[v]` lần theo thứ tự `v = 1..100` để được danh sách đã sắp xếp mà **không cần so sánh**: `O(n + V)` với `V = 100`.

In [7]:
nums = [5, 4, 2, 3, 2]
cnt = [0] * 101
for x in nums:
    cnt[x] += 1
sorted_nums = []
for v in range(1, 101):
    sorted_nums.extend([v] * cnt[v])
print(sorted_nums)             # [2, 2, 3, 4, 5]

[2, 2, 3, 4, 5]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy chơi thử với `nums = [5, 4, 2, 3]`, ghi lại từng bước:

| vòng | `nums` lúc đầu | Alice lấy | Bob lấy | `arr` sau vòng |
|---|---|---|---|---|
| 1 | `[5, 4, 2, 3]` | `2` | `3` | `[3, 2]` (Bob `3`, rồi Alice `2`) |
| 2 | `[5, 4]` | `4` | `5` | `[3, 2, 5, 4]` (Bob `5`, rồi Alice `4`) |

Quan sát quan trọng: các phần tử bị lấy ra theo thứ tự **tăng dần**: `2, 3, 4, 5`. Alice lấy phần tử **thứ nhất, thứ ba, ...** (chỉ số chẵn trong dãy đã sắp xếp), Bob lấy phần tử **thứ hai, thứ tư, ...** (chỉ số lẻ). Mỗi vòng Bob (số lớn hơn) đứng trước Alice (số nhỏ hơn) nên `arr` là dãy đã sắp xếp với **mỗi cặp liền kề đổi chỗ**: `[2, 3 | 4, 5]` thành `[3, 2 | 5, 4]`.

In [8]:
nums = [5, 4, 2, 3]
s = sorted(nums)
print("đã sắp xếp:", s)
for i in range(0, len(s), 2):
    print(f"vòng {i // 2 + 1}: Alice lấy {s[i]}, Bob lấy {s[i + 1]} -> thêm [{s[i + 1]}, {s[i]}]")

đã sắp xếp: [2, 3, 4, 5]
vòng 1: Alice lấy 2, Bob lấy 3 -> thêm [3, 2]
vòng 2: Alice lấy 4, Bob lấy 5 -> thêm [5, 4]


## 5. Cách 1 — Brute Force / baseline: mô phỏng bằng `min` và `remove`

**Ý tưởng.** Làm đúng như đề: sao chép `nums`; khi còn phần tử, Alice lấy `min` rồi `remove`, Bob lấy `min` rồi `remove`, sau đó `append(bob)` rồi `append(alice)`.

In [9]:
def number_game_simulate(nums: List[int]) -> List[int]:
    rest = list(nums)                      # sao chép, không làm hỏng mảng của người gọi
    arr = []
    while rest:
        alice = min(rest)                  # Alice lấy nhỏ nhất
        rest.remove(alice)
        bob = min(rest)                    # Bob lấy nhỏ nhất của phần còn lại
        rest.remove(bob)
        arr.append(bob)                    # Bob thêm TRƯỚC
        arr.append(alice)                  # rồi mới đến Alice
    return arr


for nums, expected in examples:
    print(nums, "->", number_game_simulate(nums), "(kỳ vọng", expected, ")")

[5, 4, 2, 3] -> [3, 2, 5, 4] (kỳ vọng [3, 2, 5, 4] )
[2, 5] -> [5, 2] (kỳ vọng [5, 2] )
[1, 1, 1, 1] -> [1, 1, 1, 1] (kỳ vọng [1, 1, 1, 1] )


In [10]:
# Xem từng vòng chơi của ví dụ 1
rest, arr, round_no = [5, 4, 2, 3], [], 0
while rest:
    round_no += 1
    alice = min(rest); rest.remove(alice)
    bob = min(rest); rest.remove(bob)
    arr.append(bob); arr.append(alice)
    print(f"vòng {round_no}: Alice={alice} Bob={bob} | còn lại {rest} | arr={arr}")

vòng 1: Alice=2 Bob=3 | còn lại [5, 4] | arr=[3, 2]
vòng 2: Alice=4 Bob=5 | còn lại [] | arr=[3, 2, 5, 4]


**Độ phức tạp (Cách 1).** Có `n / 2` vòng, mỗi vòng hai lần `min` và hai lần `remove`, mỗi lệnh quét `O(n)`: **Time `O(n²)`**; bản sao: **Space `O(n)`** (cộng mảng kết quả, vốn là đầu ra).

**Nút thắt.** Mỗi lần "lấy nhỏ nhất" ta **quét lại cả danh sách**. Nhưng thứ tự lấy ra chính là thứ tự **đã sắp xếp**, nên ta chỉ cần sắp xếp **một lần**.

## 6. Ý tưởng thuật toán / bước đột phá

Việc "luôn lấy phần tử nhỏ nhất còn lại" cho ra các phần tử theo **thứ tự tăng dần**. Nếu `s = sorted(nums)` thì các lần lấy là `s[0], s[1], s[2], ...`:

- vòng `k` (tính từ `0`): Alice lấy `s[2k]`, Bob lấy `s[2k + 1]`;
- `arr` nhận `s[2k + 1]` rồi `s[2k]`.

Vậy **`arr` = `s` với từng cặp `(s[2k], s[2k + 1])` đổi chỗ**. Không còn mô phỏng: sắp xếp `O(n log n)` rồi đổi chỗ `O(n)`. Các lựa chọn khác để có thứ tự tăng dần: **heap** (lấy dần) hoặc **counting sort** (vì giá trị `<= 100`).

## 7. Các cách tiếp theo

### Cách 2 — Sắp xếp rồi đổi chỗ từng cặp

**Giải thích.** `s = sorted(nums)`; rồi với `i = 0, 2, 4, ...` đổi `s[i]` và `s[i + 1]`. Cũng có thể viết một dòng: `s[::2], s[1::2] = s[1::2], s[::2]` (mục 3.4).

In [11]:
def number_game_sort(nums: List[int]) -> List[int]:
    s = sorted(nums)                       # sorted: không làm đổi nums
    for i in range(0, len(s), 2):
        s[i], s[i + 1] = s[i + 1], s[i]    # đổi chỗ cặp (Alice, Bob) -> (Bob, Alice)
    return s


def number_game_sort_slice(nums: List[int]) -> List[int]:
    s = sorted(nums)
    s[::2], s[1::2] = s[1::2], s[::2]      # đổi chỗ mọi cặp bằng một câu lệnh
    return s


for nums, expected in examples:
    print(nums, "->", number_game_sort(nums), number_game_sort_slice(nums), "(kỳ vọng", expected, ")")

[5, 4, 2, 3] -> [3, 2, 5, 4] [3, 2, 5, 4] (kỳ vọng [3, 2, 5, 4] )
[2, 5] -> [5, 2] [5, 2] (kỳ vọng [5, 2] )
[1, 1, 1, 1] -> [1, 1, 1, 1] [1, 1, 1, 1] (kỳ vọng [1, 1, 1, 1] )


**Độ phức tạp Cách 2:** **Time `O(n log n)`** (sắp xếp), **Space `O(n)`** cho bản sao đã sắp xếp.

### Cách 3 — Min-heap `heapq`

**Giải thích.** Đưa `nums` vào heap. Mỗi vòng `heappop` hai lần: lần đầu là Alice, lần hai là Bob; `append(bob)`, `append(alice)`. Giống mô phỏng nhưng mỗi lần lấy nhỏ nhất chỉ `O(log n)` thay vì `O(n)`.

In [12]:
def number_game_heap(nums: List[int]) -> List[int]:
    heap = list(nums)
    heapq.heapify(heap)                    # O(n)
    arr = []
    while heap:
        alice = heapq.heappop(heap)        # nhỏ nhất
        bob = heapq.heappop(heap)          # nhỏ nhất tiếp theo
        arr.append(bob)
        arr.append(alice)
    return arr


for nums, expected in examples:
    print(nums, "->", number_game_heap(nums), "(kỳ vọng", expected, ")")

[5, 4, 2, 3] -> [3, 2, 5, 4] (kỳ vọng [3, 2, 5, 4] )
[2, 5] -> [5, 2] (kỳ vọng [5, 2] )
[1, 1, 1, 1] -> [1, 1, 1, 1] (kỳ vọng [1, 1, 1, 1] )


**Độ phức tạp Cách 3:** `heapify` `O(n)`, `n` lần `heappop` mỗi lần `O(log n)`: **Time `O(n log n)`**, **Space `O(n)`**.

### Cách 4 — Counting sort (vì `1 <= nums[i] <= 100`)

**Giải thích.** Đếm `cnt[v]`, dựng danh sách đã sắp xếp bằng cách lặp `v` từ `1` đến `100` và thêm `v` đúng `cnt[v]` lần, rồi đổi chỗ từng cặp. Tổng `O(n + V)`, không dùng so sánh.

In [13]:
def number_game_counting(nums: List[int]) -> List[int]:
    cnt = [0] * 101
    for x in nums:
        cnt[x] += 1
    s = []
    for v in range(1, 101):
        s.extend([v] * cnt[v])             # v xuất hiện cnt[v] lần, theo thứ tự tăng dần
    for i in range(0, len(s), 2):
        s[i], s[i + 1] = s[i + 1], s[i]
    return s


for nums, expected in examples:
    print(nums, "->", number_game_counting(nums), "(kỳ vọng", expected, ")")

[5, 4, 2, 3] -> [3, 2, 5, 4] (kỳ vọng [3, 2, 5, 4] )
[2, 5] -> [5, 2] (kỳ vọng [5, 2] )
[1, 1, 1, 1] -> [1, 1, 1, 1] (kỳ vọng [1, 1, 1, 1] )


**Độ phức tạp Cách 4:** **Time `O(n + V)`** với `V = 100`, **Space `O(n + V)`**.

In [14]:
# Code nộp LeetCode cho Cách 2 (ngắn, rõ, O(n log n))
from typing import List

class Solution:
    def numberGame(self, nums: List[int]) -> List[int]:
        s = sorted(nums)
        for i in range(0, len(s), 2):
            s[i], s[i + 1] = s[i + 1], s[i]
        return s

## 8. TRACE từng bước

Ta theo dõi (a) Cách 2 (sắp xếp + đổi chỗ) với `[5, 4, 2, 3, 9, 7]` và (b) Cách 3 (heap) với `[5, 4, 2, 3]`.

In [15]:
def trace_sort(nums):
    s = sorted(nums)
    print("nums =", nums, "-> đã sắp xếp:", s)
    for i in range(0, len(s), 2):
        before = s[:]
        s[i], s[i + 1] = s[i + 1], s[i]
        print(f"i={i}: đổi chỗ s[{i}]={before[i]} và s[{i + 1}]={before[i + 1]} -> {s}")
    print("=> kết quả:", s)
    return s


assert trace_sort([5, 4, 2, 3, 9, 7]) == [3, 2, 5, 4, 9, 7]
assert trace_sort([2, 5]) == [5, 2]

nums = [5, 4, 2, 3, 9, 7] -> đã sắp xếp: [2, 3, 4, 5, 7, 9]
i=0: đổi chỗ s[0]=2 và s[1]=3 -> [3, 2, 4, 5, 7, 9]
i=2: đổi chỗ s[2]=4 và s[3]=5 -> [3, 2, 5, 4, 7, 9]
i=4: đổi chỗ s[4]=7 và s[5]=9 -> [3, 2, 5, 4, 9, 7]
=> kết quả: [3, 2, 5, 4, 9, 7]
nums = [2, 5] -> đã sắp xếp: [2, 5]
i=0: đổi chỗ s[0]=2 và s[1]=5 -> [5, 2]
=> kết quả: [5, 2]


In [16]:
def trace_heap(nums):
    heap = list(nums)
    heapq.heapify(heap)
    print("heap sau heapify:", heap, "(phần tử nhỏ nhất ở đầu)")
    arr, round_no = [], 0
    while heap:
        round_no += 1
        alice = heapq.heappop(heap)
        bob = heapq.heappop(heap)
        arr.append(bob)
        arr.append(alice)
        print(f"vòng {round_no}: Alice pop {alice}, Bob pop {bob} | heap còn {sorted(heap)} | arr={arr}")
    return arr


assert trace_heap([5, 4, 2, 3]) == [3, 2, 5, 4]

heap sau heapify: [2, 3, 5, 4] (phần tử nhỏ nhất ở đầu)
vòng 1: Alice pop 2, Bob pop 3 | heap còn [4, 5] | arr=[3, 2]
vòng 2: Alice pop 4, Bob pop 5 | heap còn [] | arr=[3, 2, 5, 4]


Bảng tóm tắt (Cách 2, `[5, 4, 2, 3]`):

| bước | nội dung |
|---|---|
| sắp xếp | `[2, 3, 4, 5]` |
| `i = 0` | đổi `s[0]`, `s[1]` -> `[3, 2, 4, 5]` |
| `i = 2` | đổi `s[2]`, `s[3]` -> `[3, 2, 5, 4]` |
| kết quả | `[3, 2, 5, 4]` |

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (sắp xếp rồi đổi chỗ từng cặp). Đúng class `Solution`, đúng method `numberGame(self, nums: List[int]) -> List[int]`.

In [17]:
from typing import List

class Solution:
    def numberGame(self, nums: List[int]) -> List[int]:
        arr = sorted(nums)                       # thứ tự các phần tử bị lấy ra: tăng dần
        for i in range(0, len(arr), 2):          # mỗi vòng = một cặp (Alice, Bob)
            arr[i], arr[i + 1] = arr[i + 1], arr[i]   # Bob đứng trước Alice
        return arr


sol = Solution()
for nums, expected in examples:
    print(nums, "->", sol.numberGame(nums))

[5, 4, 2, 3] -> [3, 2, 5, 4]
[2, 5] -> [5, 2]
[1, 1, 1, 1] -> [1, 1, 1, 1]


## 10. Vì sao thuật toán đúng

- **Thứ tự lấy ra là thứ tự đã sắp xếp.** Mỗi lần người chơi lấy phần tử nhỏ nhất còn lại; do đó dãy các phần tử bị lấy ra (Alice, Bob, Alice, Bob, ...) chính là `s = sorted(nums)`: `s[0], s[1], s[2], ...`. (Nếu có giá trị trùng thì lấy bản nào cũng cho cùng giá trị, nên kết quả không đổi.)
- **Vòng `k`.** Alice lấy `s[2k]` (nhỏ nhất), Bob lấy `s[2k + 1]` (nhỏ nhất của phần còn lại). `arr` thêm Bob trước rồi Alice: `s[2k + 1], s[2k]`.
- **Hệ quả.** `arr` là `s` với mỗi cặp vị trí `(2k, 2k + 1)` đổi chỗ. Độ dài chẵn nên mọi vị trí đều thuộc đúng một cặp, không bị lẻ.
- **Hoán vị của đầu vào.** `arr` có cùng các phần tử với `nums` (chỉ đổi thứ tự), nên các tính chất như tổng, đa tập hợp giá trị được giữ nguyên.
- **Kết quả duy nhất.** Với trò chơi tất định (luôn lấy nhỏ nhất) và giá trị trùng cho cùng kết quả, đáp án là duy nhất nên có thể so sánh bằng `==`.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`, `V = 100` là miền giá trị.

- **Cách 1 (mô phỏng `min`/`remove`):** Time `O(n²)`, Space `O(n)`.
- **Cách 2 (sắp xếp + đổi chỗ):** Time `O(n log n)`, Space `O(n)`.
- **Cách 3 (heap):** Time `O(n log n)`, Space `O(n)`.
- **Cách 4 (counting sort):** Time `O(n + V)`, Space `O(n + V)`.
- Ở đây `n <= 100`: mọi cách chạy tức thì. Ý nghĩa của bài: nhận ra "lấy nhỏ nhất liên tục" = **sắp xếp**.

## 12. Bộ test

In [18]:
sol = Solution()
impls = {
    "simulate": number_game_simulate,
    "sort": number_game_sort,
    "sort_slice": number_game_sort_slice,
    "heap": number_game_heap,
    "counting": number_game_counting,
    "submit": sol.numberGame,
}

for nums, expected in examples:
    for key, fn in impls.items():
        got = fn(list(nums))
        assert got == expected, (key, nums, got, expected)
print("PASS ví dụ của đề (mọi cách)")

PASS ví dụ của đề (mọi cách)


### 12.1. Các trường hợp biên

In [19]:
def check_all(nums, expected, label):
    for key, fn in impls.items():
        got = fn(list(nums))
        assert got == expected, (label, key, got, expected)
    print("PASS", label, "->", expected)


check_all([1, 2], [2, 1], "n = 2, đã sắp xếp")
check_all([2, 1], [2, 1], "n = 2, ngược thứ tự: kết quả như nhau")
check_all([7, 7], [7, 7], "hai số bằng nhau")
check_all([1, 2, 3, 4], [2, 1, 4, 3], "đã sắp xếp tăng dần")
check_all([4, 3, 2, 1], [2, 1, 4, 3], "sắp xếp giảm dần")
check_all([3, 3, 3, 3], [3, 3, 3, 3], "toàn bằng nhau")
check_all([1, 100, 1, 100], [1, 1, 100, 100], "giá trị biên 1 và 100, có trùng")
check_all([5, 1, 5, 1, 5, 1], [1, 1, 5, 1, 5, 5], "nhiều giá trị trùng")
check_all(list(range(100, 0, -1)), [v for k in range(1, 101, 2) for v in (k + 1, k)], "n = 100 giảm dần 100..1")
check_all([100] * 100, [100] * 100, "n = 100 toàn 100")

PASS n = 2, đã sắp xếp -> [2, 1]
PASS n = 2, ngược thứ tự: kết quả như nhau -> [2, 1]
PASS hai số bằng nhau -> [7, 7]
PASS đã sắp xếp tăng dần -> [2, 1, 4, 3]
PASS sắp xếp giảm dần -> [2, 1, 4, 3]
PASS toàn bằng nhau -> [3, 3, 3, 3]
PASS giá trị biên 1 và 100, có trùng -> [1, 1, 100, 100]
PASS nhiều giá trị trùng -> [1, 1, 5, 1, 5, 5]
PASS n = 100 giảm dần 100..1 -> [2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15, 18, 17, 20, 19, 22, 21, 24, 23, 26, 25, 28, 27, 30, 29, 32, 31, 34, 33, 36, 35, 38, 37, 40, 39, 42, 41, 44, 43, 46, 45, 48, 47, 50, 49, 52, 51, 54, 53, 56, 55, 58, 57, 60, 59, 62, 61, 64, 63, 66, 65, 68, 67, 70, 69, 72, 71, 74, 73, 76, 75, 78, 77, 80, 79, 82, 81, 84, 83, 86, 85, 88, 87, 90, 89, 92, 91, 94, 93, 96, 95, 98, 97, 100, 99]
PASS n = 100 toàn 100 -> [100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 10

### 12.2. Tính chất và phản ví dụ

In [20]:
# Tính chất 1: arr là hoán vị của nums
random.seed(11)
for _ in range(500):
    n = random.randrange(2, 101, 2)
    nums = [random.randint(1, 100) for _ in range(n)]
    arr = sol.numberGame(nums)
    assert sorted(arr) == sorted(nums) and len(arr) == n
print("PASS arr là hoán vị của nums")

# Tính chất 2: mỗi cặp (arr[2k], arr[2k+1]) = (Bob, Alice) nên arr[2k] >= arr[2k+1]
for _ in range(500):
    n = random.randrange(2, 101, 2)
    nums = [random.randint(1, 100) for _ in range(n)]
    arr = sol.numberGame(nums)
    assert all(arr[2 * k] >= arr[2 * k + 1] for k in range(n // 2))
    # và các cặp theo vòng không giảm: cặp sau không nhỏ hơn cặp trước
    assert all(arr[2 * k + 1] <= arr[2 * k + 2 + 1] for k in range(n // 2 - 1))
print("PASS mỗi cặp có số lớn đứng trước; các cặp tăng dần theo vòng")

# Tính chất 3: kết quả không phụ thuộc thứ tự của đầu vào
for _ in range(300):
    n = random.randrange(2, 31, 2)
    nums = [random.randint(1, 20) for _ in range(n)]
    shuffled = nums[:]
    random.shuffle(shuffled)
    assert sol.numberGame(nums) == sol.numberGame(shuffled)
print("PASS không phụ thuộc thứ tự đầu vào")

# Tính chất 4: đầu vào không bị sửa
nums = [5, 4, 2, 3]
copy = list(nums)
for key, fn in impls.items():
    fn(nums)
assert nums == copy
print("PASS đầu vào không bị sửa")

# Phản ví dụ 1: thêm Alice trước Bob
def sai_thu_tu(nums):
    s = sorted(nums)
    return s                                   # arr = [alice, bob, ...] không đổi chỗ

assert sai_thu_tu([5, 4, 2, 3]) == [2, 3, 4, 5] != [3, 2, 5, 4]
print("PASS phản ví dụ: không đổi chỗ cho", sai_thu_tu([5, 4, 2, 3]))

# Phản ví dụ 2: đổi chỗ lệch cặp (đổi mọi cặp liền kề kiểu bubble)
def sai_lech_cap(nums):
    s = sorted(nums)
    for i in range(len(s) - 1):
        s[i], s[i + 1] = s[i + 1], s[i]
    return s

assert sai_lech_cap([5, 4, 2, 3]) != [3, 2, 5, 4]
print("PASS phản ví dụ: đổi chỗ mọi cặp liền kề cho", sai_lech_cap([5, 4, 2, 3]))

PASS arr là hoán vị của nums
PASS mỗi cặp có số lớn đứng trước; các cặp tăng dần theo vòng
PASS không phụ thuộc thứ tự đầu vào
PASS đầu vào không bị sửa
PASS phản ví dụ: không đổi chỗ cho [2, 3, 4, 5]
PASS phản ví dụ: đổi chỗ mọi cặp liền kề cho [3, 4, 5, 2]


### 12.3. Đối chiếu ngẫu nhiên (có seed)

In [21]:
random.seed(2974)
for trial in range(3000):
    n = random.randrange(2, 101, 2)
    hi = random.choice([2, 5, 30, 100])
    nums = [random.randint(1, hi) for _ in range(n)]
    ref = number_game_simulate(list(nums))
    for key, fn in impls.items():
        assert fn(list(nums)) == ref, (key, nums)
print("PASS 3000 bộ ngẫu nhiên - mọi cách khớp nhau")

PASS 3000 bộ ngẫu nhiên - mọi cách khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Không đổi chỗ (thêm Alice trước Bob)

In [22]:
def sai_alice_truoc(nums):
    rest = list(nums)
    arr = []
    while rest:
        alice = min(rest); rest.remove(alice)
        bob = min(rest); rest.remove(bob)
        arr.append(alice)                          # sai thứ tự: Alice trước
        arr.append(bob)
    return arr


print("sai  :", sai_alice_truoc([5, 4, 2, 3]), "| đúng:", Solution().numberGame([5, 4, 2, 3]))

sai  : [2, 3, 4, 5] | đúng: [3, 2, 5, 4]


**Vì sao sai:** đề nói **Bob thêm vào `arr` trước, Alice thêm sau**, nên mỗi cặp phải là `[bob, alice]` chứ không phải `[alice, bob]`. Chỉ cần đảo hai lệnh `append`.

### Lỗi 2: Quên sắp xếp

In [23]:
def sai_khong_sap_xep(nums):
    arr = list(nums)
    for i in range(0, len(arr), 2):
        arr[i], arr[i + 1] = arr[i + 1], arr[i]    # đổi chỗ trên mảng chưa sắp xếp
    return arr


print("sai  :", sai_khong_sap_xep([5, 4, 2, 3]), "| đúng:", Solution().numberGame([5, 4, 2, 3]))

sai  : [4, 5, 3, 2] | đúng: [3, 2, 5, 4]


**Vì sao sai:** Alice và Bob luôn lấy **phần tử nhỏ nhất**, không phải phần tử đứng đầu mảng. Chỉ khi dãy đã sắp xếp thì "hai phần tử đầu" mới là "hai nhỏ nhất". Phải `sorted` trước khi đổi chỗ.

### Lỗi 3: Đổi chỗ lệch cặp

In [24]:
def sai_lech_cap_2(nums):
    s = sorted(nums)
    for i in range(len(s) - 1):                    # i chạy 0, 1, 2, ... thay vì 0, 2, 4, ...
        s[i], s[i + 1] = s[i + 1], s[i]
    return s


print("sai  :", sai_lech_cap_2([5, 4, 2, 3]), "| đúng:", Solution().numberGame([5, 4, 2, 3]))

sai  : [3, 4, 5, 2] | đúng: [3, 2, 5, 4]


**Vì sao sai:** cặp là `(s[0], s[1])`, `(s[2], s[3])`, ... (bước `2`). Vòng `range(n - 1)` đổi cả `(s[1], s[2])` nữa, tức di chuyển phần tử `s[0]` đến cuối như bong bóng, phá hỏng cấu trúc cặp. Dùng `range(0, n, 2)`.

### Lỗi 4: Sửa luôn mảng đầu vào bằng `nums.sort()`

In [25]:
def sai_sua_dau_vao(nums):
    nums.sort()                                   # sắp xếp ngay trên mảng của người gọi
    for i in range(0, len(nums), 2):
        nums[i], nums[i + 1] = nums[i + 1], nums[i]
    return nums


data = [5, 4, 2, 3]
res = sai_sua_dau_vao(data)
print("kết quả:", res, "| mảng gốc sau khi gọi:", data, "| cùng một đối tượng:", res is data)

kết quả: [3, 2, 5, 4] | mảng gốc sau khi gọi: [3, 2, 5, 4] | cùng một đối tượng: True


**Vì sao không tốt:** `nums.sort()` và đổi chỗ trực tiếp **làm hỏng mảng của người gọi**, và còn trả về **cùng đối tượng** nên người gọi giữ hai tên cho một mảng đã bị đổi. Trong một bài kiểm tra tự động (hoặc khi bạn cần dùng lại `nums`), kết quả sẽ sai. Dùng `sorted(nums)` để có bản sao.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "mỗi lượt lấy **phần tử nhỏ nhất/lớn nhất** còn lại" → **sắp xếp một lần** (hoặc dùng heap khi dữ liệu thay đổi giữa các lượt).
- Công thức nhớ: **`s = sorted(nums)`; đổi chỗ `(s[2k], s[2k+1])`**; hoặc `s[::2], s[1::2] = s[1::2], s[::2]`.
- Dạng "hai người lấy xen kẽ" → người thứ nhất lấy chỉ số chẵn, người thứ hai lấy chỉ số lẻ **của dãy đã sắp xếp**.
- Giá trị nhỏ (`<= 100`) → **counting sort** `O(n + V)`.
- `nums.sort()` sửa tại chỗ và trả về `None`; `sorted(nums)` trả về bản sao. Chọn đúng để không có tác dụng phụ.
- Cẩn thận thứ tự `append` khi đề nêu rõ "người nào thêm trước".

## 15. Biến thể của bài

1. Đảo vai: **Bob lấy trước**, hoặc Alice thêm trước.
2. Mỗi người lấy **phần tử lớn nhất** thay vì nhỏ nhất.
3. Có **ba người** chơi xen kẽ (mỗi vòng lấy ba số nhỏ nhất).
4. Trả về **tổng các phần tử Alice lấy** và **tổng Bob lấy**.
5. Độ dài `nums` có thể **lẻ**: người cuối cùng của vòng cuối lấy phần tử còn lại duy nhất.

Thử biến thể 2, 4 và 5:

In [26]:
def number_game_max(nums):
    s = sorted(nums, reverse=True)                  # lấy lớn nhất liên tục
    for i in range(0, len(s) - 1, 2):
        s[i], s[i + 1] = s[i + 1], s[i]
    return s


def alice_bob_sums(nums):
    s = sorted(nums)
    return sum(s[::2]), sum(s[1::2])                # Alice: chỉ số chẵn; Bob: chỉ số lẻ


def number_game_odd(nums):
    s = sorted(nums)
    for i in range(0, len(s) - 1, 2):               # bỏ cặp lẻ cuối cùng nếu độ dài lẻ
        s[i], s[i + 1] = s[i + 1], s[i]
    return s


print("2.", number_game_max([5, 4, 2, 3]))            # [4, 5, 2, 3]
print("4.", alice_bob_sums([5, 4, 2, 3]))             # (6, 8): Alice 2+4, Bob 3+5
print("5.", number_game_odd([5, 4, 2, 3, 1]))         # [2, 1, 4, 3, 5]
assert number_game_max([5, 4, 2, 3]) == [4, 5, 2, 3]
assert alice_bob_sums([5, 4, 2, 3]) == (6, 8)
assert number_game_odd([5, 4, 2, 3, 1]) == [2, 1, 4, 3, 5]
assert number_game_odd([5, 4, 2, 3]) == Solution().numberGame([5, 4, 2, 3])

2. [4, 5, 2, 3]
4. (6, 8)
5. [2, 1, 4, 3, 5]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng `min` + `remove` | làm đúng như đề | `O(n²)` | `O(n)` | hiểu đề, đáp án chuẩn |
| 2. Sắp xếp + đổi chỗ từng cặp | thứ tự lấy = thứ tự sắp xếp | `O(n log n)` | `O(n)` | **cách nộp bài** |
| 3. Min-heap `heapq` | `heappop` hai lần mỗi vòng | `O(n log n)` | `O(n)` | khi vừa lấy vừa thêm phần tử |
| 4. Counting sort | đếm theo giá trị `<= 100` | `O(n + V)` | `O(n + V)` | giá trị nhỏ, cần tuyến tính |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `sorted([3, 1, 2])`, `[1, 2, 3, 4, 5, 6][::2]`, `[1, 2, 3, 4, 5, 6][1::2]`, `Solution().numberGame([4, 1])`, `Solution().numberGame([6, 2, 9, 3])`.

**Bài 2 (tính tay).** Với `nums = [8, 3, 6, 1, 7, 2]`: sắp xếp, ghi từng vòng (Alice lấy gì, Bob lấy gì, `arr` sau vòng) và cho biết `arr` cuối.

**Bài 3 (điền TODO).** Hoàn thành `number_game_todo(nums)` theo Cách 2.

In [27]:
def number_game_todo(nums):
    # TODO 1: s = bản sao đã sắp xếp của nums (dùng sorted, không dùng nums.sort())
    # TODO 2: với i = 0, 2, 4, ...: đổi chỗ s[i] và s[i + 1]
    # TODO 3: trả về s
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `number_game_heap2(nums)` bằng **heap** (Cách 3) mà không nhìn code mẫu. Kiểm tra với ba ví dụ trong `examples`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `number_game_bob_first(nums)`: Bob lấy trước rồi Alice (Alice thêm vào `arr` trước, rồi Bob). Kết quả mong đợi với `[5, 4, 2, 3]` là gì? (b) Viết `rounds(nums)` trả về **danh sách các bộ `(alice, bob)`** theo vòng. (c) Viết `number_game_k(nums, k)` tổng quát: mỗi vòng `k` người lấy `k` số nhỏ nhất; người thứ `k` thêm vào `arr` đầu tiên, tiếp theo là người `k - 1`, ..., người `1` thêm cuối (`len(nums)` chia hết cho `k`). Với `k = 2` phải trùng đáp án gốc.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[1, 2, 3]`, `[1, 3, 5]`, `[2, 4, 6]`, `[4, 1]`, `[3, 2, 9, 6]`. Bài 3: `s = sorted(nums)`; `for i in range(0, len(s), 2): s[i], s[i + 1] = s[i + 1], s[i]`; `return s`.
- **Mức 2 (vừa):** Bài 2: sắp xếp `[1, 2, 3, 6, 7, 8]`. Vòng 1: Alice `1`, Bob `2` -> `[2, 1]`; vòng 2: Alice `3`, Bob `6` -> `[2, 1, 6, 3]`; vòng 3: Alice `7`, Bob `8` -> `[2, 1, 6, 3, 8, 7]`. Bài 4: `heapify`, `while heap:` rồi hai `heappop`.
- **Mức 3 (sâu):** Bài 5(a): Bob lấy trước nên Bob nhận phần tử nhỏ hơn: với `[2, 3, 4, 5]` Bob lấy `2`, Alice lấy `3`; Alice thêm trước nên `arr = [3, 2, 5, 4]`, trùng đáp án gốc. 5(b): `list(zip(s[::2], s[1::2]))`. 5(c): sắp xếp rồi chia thành nhóm `k` phần tử liên tiếp, **đảo ngược mỗi nhóm** `group[::-1]`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1
assert sorted([3, 1, 2]) == [1, 2, 3]
assert [1, 2, 3, 4, 5, 6][::2] == [1, 3, 5] and [1, 2, 3, 4, 5, 6][1::2] == [2, 4, 6]
assert Solution().numberGame([4, 1]) == [4, 1]
assert Solution().numberGame([6, 2, 9, 3]) == [3, 2, 9, 6]

# Bài 2
assert Solution().numberGame([8, 3, 6, 1, 7, 2]) == [2, 1, 6, 3, 8, 7]

# Bài 3
def number_game_todo_ans(nums):
    s = sorted(nums)
    for i in range(0, len(s), 2):
        s[i], s[i + 1] = s[i + 1], s[i]
    return s

for nums, expected in examples:
    assert number_game_todo_ans(nums) == expected

# Bài 4
def number_game_heap2(nums):
    heap = list(nums)
    heapq.heapify(heap)
    arr = []
    while heap:
        alice = heapq.heappop(heap)
        bob = heapq.heappop(heap)
        arr += [bob, alice]
    return arr

for nums, expected in examples:
    assert number_game_heap2(nums) == expected

# Bài 5(a)
def number_game_bob_first(nums):
    rest = list(nums)
    arr = []
    while rest:
        bob = min(rest); rest.remove(bob)
        alice = min(rest); rest.remove(alice)
        arr += [alice, bob]                         # Alice thêm trước
    return arr

assert number_game_bob_first([5, 4, 2, 3]) == [3, 2, 5, 4]
assert number_game_bob_first([2, 5]) == [5, 2]

# Bài 5(b)
def rounds(nums):
    s = sorted(nums)
    return list(zip(s[::2], s[1::2]))

assert rounds([5, 4, 2, 3]) == [(2, 3), (4, 5)]

# Bài 5(c)
def number_game_k(nums, k):
    s = sorted(nums)
    out = []
    for i in range(0, len(s), k):
        out += s[i:i + k][::-1]                      # người thứ k thêm đầu tiên ... người 1 thêm cuối
    return out

assert number_game_k([5, 4, 2, 3], 2) == [3, 2, 5, 4]
assert number_game_k([5, 4, 2, 3, 9, 7], 3) == [4, 3, 2, 9, 7, 5]
for _ in range(100):
    n = random.randrange(2, 41, 2)
    arr = [random.randint(1, 50) for _ in range(n)]
    assert number_game_k(arr, 2) == Solution().numberGame(arr)
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (sắp xếp một lần, đổi chỗ từng cặp)

```python
from typing import List

class Solution:
    def numberGame(self, nums: List[int]) -> List[int]:
        arr = sorted(nums)
        for i in range(0, len(arr), 2):
            arr[i], arr[i + 1] = arr[i + 1], arr[i]
        return arr
```

**Checklist**

- [ ] Đã **sắp xếp** trước (người chơi luôn lấy phần tử nhỏ nhất, không phải phần tử đầu mảng)?
- [ ] Đổi chỗ theo từng **cặp** (`range(0, n, 2)`), không phải mọi cặp liền kề?
- [ ] Thứ tự thêm: **Bob trước, Alice sau** trong mỗi vòng?
- [ ] Dùng `sorted(nums)` (bản sao), không dùng `nums.sort()` làm hỏng đầu vào?
- [ ] Đã thử: `n = 2`, giá trị trùng, mảng đã sắp xếp, mảng giảm dần, `n = 100`?
- [ ] Skeleton đúng `class Solution` / `def numberGame(self, nums: List[int]) -> List[int]`?